In [28]:
# !pip install duckdb
# !pip install sqlalchemy
# !pip install pyodbc
# !pip install pandas
# !pip install polars


In [37]:
import datetime 
from MODULOS.conexion_sql import get_connection
from MODULOS.conexion_sql import get_sqlalchemy_engine
import duckdb
import pandas as pd
from datetime import datetime
from matplotlib.dates import relativedelta
from pathlib import Path
import polars as pl

hora_inicio = datetime.now()

RUTA_PRINCIPAL = f"D:\\DATOS\\dataset"


In [38]:
def SQLServer_to_Parquet(ScriptSQL, ruta_parquet):
    
    conn = get_connection()
    if conn is None:
        print("No se pudo conectar a SQL Server")
        return
    try:
        query = f"""
                {ScriptSQL}
        """
        # df = pd.read_sql(query, conn)
        df = pd.read_sql(query, conn)
        print(f"Registros obtenidos: {len(df):,}")
        duckdb.sql(f"""
            COPY (
                SELECT *
                FROM df
            )
            TO '{ruta_parquet}'
            (
                FORMAT PARQUET,
                COMPRESSION ZSTD
            )
        """)
        print("Parquet generado correctamente")
        print(ruta_parquet)
    except Exception as e:
        print(f"Error: {e}")
    finally:
        # print("Cerrando conexión a SQL Server")
        conn.close()

In [34]:
TABLA = "FAC.MBCORE_GUIAS"
RUTA_PARQUET = f"{RUTA_PRINCIPAL}\\{TABLA}.parquet"
print(f"Ruta del archivo Parquet: {RUTA_PARQUET}")

SCRIPT_SQL = f"""
;WITH CTE AS( SELECT 
		           [GUIA],[TIME_ID] =  convert(int,convert(datetime,convert(date,convert(datetime,CASE WHEN [FECHA_INICIO]='NO DISPONIBLE' OR  [FECHA_INICIO] IS NULL THEN CONVERT(DATETIME,'1900-01-01')  ELSE [FECHA_INICIO] END ))))
                  ,[FECHA_INICIO] =  CASE WHEN [FECHA_INICIO]='NO DISPONIBLE' OR  [FECHA_INICIO] IS NULL THEN CONVERT(DATETIME,'1900-01-01')  ELSE [FECHA_INICIO] END
                  ,[NUMERO_PIEZAS],[TIEMPO_ENTREGA],[NOMBRE_TIEMPO_ENTREGA],[ID_PRODUCTO],[PRODUCTO],[DIRECCION_DESTINO],[ID_CIUDAD_ORIGEN],[CIUDAD_ORIGEN],[DEPARTAMENTO_ORIGEN],[REGIONAL_ORIGEN]
                  ,[ID_CIUDAD_DESTINO],[CIUDAD_DESTINO],[DEPARTAMENTO_DESTINO],[REGIONAL_DESTINO],[ID_CLIENTE],[NOMBRE_CLIENTE],[ID_ESTADO_ENVIO],[ESTADO_ENVIO],[ID_PAIS_UBICACION]
                  ,[ID_CIUDAD_UBICACION],[ID_DIVISION_UBICACION],[VALOR_DECLARADO],[ID_BODEGA_ULTM_MOVI],[CANAL] = IIF([CANAL] LIKE '%DISPONIB%','OTROS',[CANAL])
                  ,[ID_CIUDAD_CANAL_ORIGEN],[CIUDAD_CANAL_ORIGEN],[DEPARTAMENTO_CANAL_ORIGEN],[REGIONAL_CANAL_ORIGEN]
                  ,[ID_CIUDAD_CANAL_DESTINO],[CIUDAD_CANAL_DESTINO],[DEPARTAMENTO_CANAL_DESTINO],[REGIONAL_CANAL_DESTINO],[SE_ID_FECHA_EZU],[SE_FECHA_EZU],[SE_ID_FECHA_ULTI_MOVI],[SE_FECHA_ULTI_MOVI],[SE_CANT_SZU],[SE_ID_FECHA_SZU],[SE_FECHA_SZU]
                  ,[SE_ID_CONC_SZU],[SE_CONC_SZU],[SE_ID_CANT_DZU],[SE_ID_ULTI],[SE_ID_SZU],[ID_CIUDAD_ULTI_MOVI],[CIUDAD_ULTI_MOVI],[DEPTARTAMENTO_ULTI_MOVI],[REGIONAL_ULTI_MOVI],[ID_ZONA],[PROCESO_ULTI_MOVI],[CONCECUTIVO_ULTI_MOVI],[BODEGA_ULTM_MOVI],[ID_TIPO_TRAYECTO]
                  ,[TIPO_TRAYECTO],[ID_CONFIRMACION],[CANT_CONFIRMACION],[CONFIRMACION_SOLUCION_1],[CONFIRMACION_SOLUCION_2],[CONFIRMACION_SOLUCION_3],[DEVOLUCION_1],[DEVOLUCION_2],[DEVOLUCION_3],[NOVEDAD_OPERATIVA_1]
                  ,[NOVEDAD_OPERATIVA_2],[NOVEDAD_OPERATIVA_3],[ENVIO_TAREA],[ESTADO_TAREA],[ID_PROC_ULTI_MOVI],[ID_CONC_ULTI_MOVI],[PESO_FISICO_R],[PESO_FISICO_E]
                  ,[TELEFONO_DESTINATARIO],[REGIMEN] = CASE WHEN UPPER([REGIMEN]) LIKE '%MENSAJER%EXPRES%' THEN 'Mensajeria Expresa' ELSE [REGIMEN] END
                  ,[ESTADOFISICO],[FECHA_BOD_Rezagos],[FECHA_BOD_Compensaciones],[Numero_veces_compensaicones] = IIF([Numero_veces_compensaicones]=0 AND CR_ID_APLICA_COMPENSACIONES=1,1,[Numero_veces_compensaicones])
                  ,[Numero_veces_rezagos],[Numero_piezas_rezagos],[Numero_piezas_compensacion],[ID_ESTADO_INDEMNIZACION]
                  ,[ESTADO_INDEMNIZACION],[DURACION_COMPENSACIONES],[DURACION_REZAGOS],[VALIDACION_SE_FECHA_ULTI_MOVI],[VALIDACION_G_ID_FECHA],[VEJEZ_DESDE_INICIO] = [VEJEZ_DESDE_INICIO] +DATEDIFF("DAY",CASE WHEN  [FECHA_EXTRACCION_DM] IS NULL THEN CONVERT(DATETIME,'1900-01-01') ELSE [FECHA_EXTRACCION_DM] END,GETDATE())
	              ,[VEJEZ A HOY] = DATEDIFF("DAY",CASE WHEN [FECHA_INICIO]='NO DISPONIBLE' OR  [FECHA_INICIO] IS NULL THEN CONVERT(DATETIME,'1900-01-01')  ELSE [FECHA_INICIO] END,GETDATE())
                  ,[FECHA_EXTRACCION_DM],BODEGA_UBICACION = CASE WHEN  [CR_BODEGA_UBICACION] IS NULL THEN BODEGA_ULTM_MOVI ELSE [CR_BODEGA_UBICACION] END
                  ,REGIONAL_UBICACION = CASE WHEN   [CR_REGIONAL_UBICACION] IS NULL THEN REGIONAL_ULTI_MOVI ELSE [CR_REGIONAL_UBICACION] END
	            ,[VALOR_FLETES_GUIAS] ,[CR_ID_APLICA_COMPENSACIONES],[TAR_PROM_TIEMPO_RESPUESTA] = ROUND(CONVERT(FLOAT,[TAR_PROM_TIEMPO_RESPUESTA])/60,2)
	            ,[TAR_PROM_TIEMPO_CORRIDO] = ROUND(CONVERT(FLOAT,[TAR_PROM_TIEMPO_CORRIDO])/60,2)
	            ,[TAR_PROM_TIEMPO_TOTAL] = ROUND(CONVERT(FLOAT,[TAR_PROM_TIEMPO_TOTAL]) /60,2)
	            ,[TAR_NUM_TAREAS],[TAR_NUM_TAREAS_CERRADA],[TAR_NUM_TAREAS_PENDIENTE] 
              FROM [FAC].[MBCORE_GUIAS] WITH(NOLOCK) 
WHERE TIME_ID > CONVERT(INT,DATEADD(yy,-1,DATEADD(yy,DATEDIFF(yy,0,GETDATE()),0)))
)
SELECT * FROM CTE WHERE LEN(GUIA)>2 AND CR_ID_APLICA_COMPENSACIONES =1
OPTION(RECOMPILE)
"""

SQLServer_to_Parquet(
    SCRIPT_SQL,
    RUTA_PARQUET
)
print(f"Finaliza carga de {TABLA}: {RUTA_PARQUET}")
df_resultado = duckdb.sql(f"""
    SELECT *
    FROM '{RUTA_PARQUET}'
    LIMIT 5
""").df()
display(df_resultado)
hora_final = datetime.now()
duracion = hora_final - hora_inicio
print(f"Duración de la ejecución: {duracion}")

Ruta del archivo Parquet: D:\DATOS\dataset\FAC.MBCORE_GUIAS.parquet
Conexión establecida con SQL Server


C:\Users\john.cruz\AppData\Local\Temp\ipykernel_22568\2566400272.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


Registros obtenidos: 92,005
Parquet generado correctamente
D:\DATOS\dataset\FAC.MBCORE_GUIAS.parquet
Finaliza carga de FAC.MBCORE_GUIAS: D:\DATOS\dataset\FAC.MBCORE_GUIAS.parquet


,GUIA,TIME_ID,FECHA_INICIO,NUMERO_PIEZAS,TIEMPO_ENTREGA,NOMBRE_TIEMPO_ENTREGA,ID_PRODUCTO,PRODUCTO,DIRECCION_DESTINO,ID_CIUDAD_ORIGEN,...,BODEGA_UBICACION,REGIONAL_UBICACION,VALOR_FLETES_GUIAS,CR_ID_APLICA_COMPENSACIONES,TAR_PROM_TIEMPO_RESPUESTA,TAR_PROM_TIEMPO_CORRIDO,TAR_PROM_TIEMPO_TOTAL,TAR_NUM_TAREAS,TAR_NUM_TAREAS_CERRADA,TAR_NUM_TAREAS_PENDIENTE
0,9.172444e+09,46090,2026-03-11 10:37:31.503,1,1,NORMAL,2,MERCANCIA PREMIER,RESTREPO BALCONES PLAZA - C CIAL BALCONES PLAZ...,10,...,COMPENSACIONES,BOGOTA,19000.0,1,0.0,0.0,0.0,0.0,0.0,0.0
1,9.189418e+09,46085,2026-03-06 19:42:05.483,1,1,NORMAL,2,MERCANCIA PREMIER,CALLE1 A # 18 - 19 TIENDA LA NUEVECITA,10,...,COMPENSACIONES,BOGOTA,19000.0,1,0.0,0.0,0.0,0.0,0.0,0.0
2,9.189418e+09,46099,2026-03-20 16:46:48.247,1,1,NORMAL,2,MERCANCIA PREMIER,CLL 77 SUR # 50 A -184 APTO 420 VIÑA DEL MAR B...,10,...,COMPENSACIONES,BOGOTA,17800.0,1,0.0,0.0,0.0,0.0,0.0,0.0
3,2.278395e+09,46201,2026-06-30 10:16:00.000,1,1,NORMAL,2,MERCANCIA PREMIER,CRA 64 C # 88 -02 AUTOPISTA NORTE,10,...,COMPENSACIONES BOGOTA,BOGOTA,30204.0,1,0.0,0.0,0.0,0.0,0.0,0.0
4,9.185631e+09,46133,2026-04-23 18:10:30.570,1,1,NORMAL,2,MERCANCIA PREMIER,CLLE 3A 22-32 BARRIOLIBERTADORES,10,...,COMPENSACIONES,BOGOTA,17800.0,1,0.0,0.0,0.0,0.0,0.0,0.0


Duración de la ejecución: 0:32:00.966389


In [40]:
TABLA = "FAC.ATENCION_CLIENTE_ESTADO_TAREAS"
RUTA_PARQUET = f"{RUTA_PRINCIPAL}\\{TABLA}.parquet"
print(f"Ruta del archivo Parquet: {RUTA_PARQUET}")

SCRIPT_SQL = f"""
SELECT 
		TIME_ID,FECHA
		,DIA,MES
		,AÑO,NUMERO_GUIA = CONVERT([numeric](20, 0),SIG_COLOMBIA_DW.[dbo].[Limpiar_Carcateres](TAR.NUMERO_DOC_PROBLEMA, '^0-9')) 	
		,DV.NOMBRE_DIVISION,TAR.ID_TIPO_IDP,TAR.ID_EST_TAREA,TAR.ID_EMPLEADO,TAR.ID_EMPLEADO_ASIGNADO,TAR.ID_PAIS
		,TAR.ID_TIPO_SERVICIO,TAR.ID_CIUDAD,TAR.ID_CIUDAD_PERTENECE,TAR.ID_DIVISION,TAR.ID_TIPO_TAREA,TAR.ID_RESPUESTA_TAREA
		,TAR.RESPONSABLE_TIPO_IDP,TAR.RESPONSABLE_PAIS,TAR.RESPONSABLE_CIUDAD,TAR.RESPONSABLE_TIPO,TAR.RESPONSABLE_CODIGO,TAR.ID_PAIS_CONTACTO
		,TAR.ID_CIUDAD_CONTACTO,TAR.ID_CONTACTO_LLAMADA,TAR.ID_ORIGEN_TAREA,TAR.TIEMPO_RESPUESTA,TAR.TIEMPO_ASIGNACION,TAR.TIEMPO_MODIFICACION
		,TAR.TIEMPO_CORRIDO,TAR.TIEMPO_TOTAL,TAR.NUM_TAREAS,TAR.ID_TIPO_VINCULACION,TAR.ID_TIPO_DIVISION_EMPL,TAR.ID_CARGO
		,TAR.ID_CIUDAD_EMPL,TAR.ID_DIVISION_EMPL,TAR.ID_DIVISION_EMPL_ASIGNADO,TAR.FECHA_RESPUESTA_TAREA
		,TAR.FECHA_MODIFICACION,TAR.FECHA_CREACION_TAREA,TAR.NUM_TAREAS_CON_RESPUESTA
 FROM SIG_COL_LOGISTICA_DW.FAC.ATENCION_CLIENTE_ESTADO_TAREAS TAR
 INNER JOIN SIG_COL_LOGISTICA_DW.DIM.DIVISIONES DV ON DV.ID_CIUDAD=TAR.ID_CIUDAD AND DV.ID_DIVISION=TAR.ID_DIVISION
 WHERE ISNUMERIC(TAR.NUMERO_DOC_PROBLEMA)=1
 AND TIME_ID>=CONVERT(INT,DATEADD(yy,-1,DATEADD(yy,DATEDIFF(yy,0,GETDATE()),0)))
"""

SQLServer_to_Parquet(
    SCRIPT_SQL,
    RUTA_PARQUET
)
print(f"Finaliza carga de {TABLA}: {RUTA_PARQUET}")
df_resultado = duckdb.sql(f"""
    SELECT *
    FROM '{RUTA_PARQUET}'
    LIMIT 5
""").df()
display(df_resultado)
hora_final = datetime.now()
duracion = hora_final - hora_inicio
print(f"Duración de la ejecución: {duracion}")

Ruta del archivo Parquet: D:\DATOS\dataset\FAC.ATENCION_CLIENTE_ESTADO_TAREAS.parquet
Conexión establecida con SQL Server


C:\Users\john.cruz\AppData\Local\Temp\ipykernel_22568\2566400272.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


Registros obtenidos: 1,927,152
Parquet generado correctamente
D:\DATOS\dataset\FAC.ATENCION_CLIENTE_ESTADO_TAREAS.parquet
Finaliza carga de FAC.ATENCION_CLIENTE_ESTADO_TAREAS: D:\DATOS\dataset\FAC.ATENCION_CLIENTE_ESTADO_TAREAS.parquet


,TIME_ID,FECHA,DIA,MES,AÑO,NUMERO_GUIA,NOMBRE_DIVISION,ID_TIPO_IDP,ID_EST_TAREA,ID_EMPLEADO,...,ID_TIPO_VINCULACION,ID_TIPO_DIVISION_EMPL,ID_CARGO,ID_CIUDAD_EMPL,ID_DIVISION_EMPL,ID_DIVISION_EMPL_ASIGNADO,FECHA_RESPUESTA_TAREA,FECHA_MODIFICACION,FECHA_CREACION_TAREA,NUM_TAREAS_CON_RESPUESTA
0,46033,2026-01-13,13,1,2026,2.277329e+09,LOGISTICA,821.0,0,1014220207,...,1,9947,10001,10,9947,1004009300,2026-01-13 11:09:49.400,2026-01-13 11:52:40.930,2026-01-13 09:34:15.837,1.0
1,46033,2026-01-13,13,1,2026,2.277329e+09,LOGISTICA,821.0,0,1014220207,...,1,9947,10001,10,9947,1005009300,2026-01-19 14:40:53.010,2026-01-19 14:55:40.420,2026-01-13 16:46:42.743,1.0
2,46033,2026-01-13,13,1,2026,2.277329e+09,LOGISTICA,821.0,0,1014220207,...,1,9947,10001,10,9947,1001509300,2026-01-13 14:25:15.827,2026-01-13 17:03:40.380,2026-01-13 12:09:10.697,1.0
3,46033,2026-01-13,13,1,2026,2.277329e+09,LOGISTICA,821.0,0,1014220207,...,1,9947,10001,10,9947,1004109300,2026-01-14 10:25:15.153,2026-01-14 11:27:23.503,2026-01-13 16:10:38.927,1.0
4,46033,2026-01-13,13,1,2026,2.277329e+09,LOGISTICA,821.0,0,1014220207,...,1,9947,10001,10,9947,1005509300,2026-01-14 10:45:28.603,2026-01-14 11:27:48.967,2026-01-13 16:11:10.443,1.0


Duración de la ejecución: 0:21:01.271175


In [35]:
TABLA = "DIM.PROCESO_TAREAS"
RUTA_PARQUET = f"{RUTA_PRINCIPAL}\\{TABLA}.parquet"
print(f"Ruta del archivo Parquet: {RUTA_PARQUET}")

SCRIPT_SQL = f"""
WITH CTE_DIM_PROCESO_TAREAS AS (
	 SELECT 
	 ROW_NUMBER ( )   OVER ( PARTITION BY NUMERO_GUIA order by NOMBRE_DIVISION desc ) ID
	 ,NUMERO_GUIA,ID_DIVISION,NOMBRE_DIVISION
	FROM [DIM].[MBCORE_PROCESO_TAREAS] WHERE LEN(NUMERO_GUIA)>4
)SELECT  
ID,NUMERO_GUIA,ID_DIVISION,NOMBRE_DIVISION FROM CTE_DIM_PROCESO_TAREAS WHERE ID=1
"""
SQLServer_to_Parquet(
    SCRIPT_SQL,
    RUTA_PARQUET
)
print(f"Finaliza carga de {TABLA}: {RUTA_PARQUET}")
df_resultado = duckdb.sql(f"""
    SELECT *
    FROM '{RUTA_PARQUET}'
    LIMIT 5
""").df()
display(df_resultado)
hora_final = datetime.now()
duracion = hora_final - hora_inicio
print(f"Duración de la ejecución: {duracion}")

Ruta del archivo Parquet: D:\DATOS\dataset\DIM.PROCESO_TAREAS.parquet
Conexión establecida con SQL Server


C:\Users\john.cruz\AppData\Local\Temp\ipykernel_22568\2566400272.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


Registros obtenidos: 4,507,260
Parquet generado correctamente
D:\DATOS\dataset\DIM.PROCESO_TAREAS.parquet
Finaliza carga de DIM.PROCESO_TAREAS: D:\DATOS\dataset\DIM.PROCESO_TAREAS.parquet


,ID,NUMERO_GUIA,ID_DIVISION,NOMBRE_DIVISION
0,1,1430053.0,9300,LOGISTICA
1,1,1513724.0,9300,LOGISTICA
2,1,35246108.0,9300,LOGISTICA
3,1,212535338.0,9300,LOGISTICA
4,1,216400318.0,9300,LOGISTICA


Duración de la ejecución: 1:20:47.369582


In [32]:
# anio = int(str(PERIODO)[:4])
# mes = int(str(PERIODO)[4:])
# # Crear el objeto datetime directamente de forma limpia
# fecha_ini = datetime(anio, mes, 1) 
# fecha_final = (fecha_ini + relativedelta(months=1) ) #- relativedelta(day=1)
# fecha_final = fecha_final - relativedelta(days=1)
# print(f"Fecha Inicial: {fecha_ini} | Fecha Final: {fecha_final}")


# ########################################  RUTAS ARCHIVO PARQUET ########################################
# PATH_PRINCIPAL = "D:/Datos"
# NOMBRE_TABLA = f"FAC.COSTO_DARWIN_BASE_OPERACION_{PERIODO}"

# # Definimos la ruta de la carpeta como un objeto Path para poder usar .mkdir()
# DIRECTORIO_PARQUET = Path(f"{PATH_PRINCIPAL}/BACKUP_FAC_COSTOS_POR_TRAYECTO/{PERIODO}")

# # Creamos la ruta (carpeta y subcarpetas) si no existe
# DIRECTORIO_PARQUET.mkdir(parents=True, exist_ok=True)

# # Ruta completa del archivo uniendo el directorio y el nombre del archivo
# RUTA_PARQUET = DIRECTORIO_PARQUET / NOMBRE_TABLA
# ########################################################################################################

# print("Directorio creado/verificado:", DIRECTORIO_PARQUET)



In [33]:

# PERIODO_INICIAL = [str(fecha_ini).replace('-', '')[:8]]
# PERIODO_FINAL = [str(fecha_final).replace('-', '')[:8]]
# print(PERIODO_INICIAL,'|',PERIODO_FINAL)

# RUTA_PARQUET_final = ""

# print(int(PERIODO_FINAL[0]))

# for date01 in range(int(PERIODO_INICIAL[0]), int(PERIODO_FINAL[0]),10):  

#     RUTA_PARQUET_final = f"{RUTA_PARQUET}_{date01}.parquet"
    
#     fecha_inicial_aux = datetime(
#             int(str(date01)[:4]), 
#             int(str(date01)[4:6]), 
#             int(str(date01)[6:8]))
        
#     fecha_final_aux = datetime(
#         int(str(date01 )[:4]), 
#         int(str(date01 )[4:6]), 
#         int(str(date01 )[6:8])) + relativedelta(days=10)

#     SCRIPT_SQL = ""

#     if fecha_final_aux + relativedelta(days=1) > fecha_final:
#         SCRIPT_SQL = f"""
#         SELECT 
#         * 
#         FROM [FAC].[COSTO_DARWIN_BASE_OPERACION_MES] WITH(NOLOCK) WHERE PERIODO_EJECUCION = {PERIODO} 
#         AND  FECHA_HORA_MOVIMIENTO BETWEEN '{fecha_inicial_aux}' AND '{fecha_final + relativedelta(days=1)}'
#         OPTION(RECOMPILE)
#         """
#         print(SCRIPT_SQL)
#     else:
#         SCRIPT_SQL = f"""
#         SELECT 
#         * 
#         FROM [FAC].[COSTO_DARWIN_BASE_OPERACION_MES] WITH(NOLOCK) WHERE PERIODO_EJECUCION = {PERIODO} 
#         AND  FECHA_HORA_MOVIMIENTO BETWEEN '{fecha_inicial_aux}' AND '{fecha_final_aux}'
#         OPTION(RECOMPILE)
#         """
#         print(SCRIPT_SQL)
    
#     SQLServer_to_Parquet(
#         SCRIPT_SQL,
#         RUTA_PARQUET_final
#     )
# print(f"Finaliza carga de {NOMBRE_TABLA}: {RUTA_PARQUET}")

#  ############################### se deja pendiente mientras sse finaliza el trabajo de darwin 